# VL04 - Text Classification with Naive Bayes
In this seminar we demonstrate the full lifecycle of training and evaluating a **Multinomial Naïve Bayes** spam classifier.
It connects the ideas from:
- **Lab 1 (rule-based spam filter)**:hand-written rules  
- **VL03 (text representation)**:Bag-of-Words model  

and shows how these come together in a *learned* classifier.

In [ ]:
import pandas as pd
import spacy
from datasets import load_from_disk
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except OSError:
    print("Warning: spaCy model 'en_core_web_sm' not found. Please run 'python -m spacy download en_core_web_sm'")
    # Fallback to a simpler model creation if the standard one fails
    nlp = spacy.blank("en")

## 1. Load and inspect the dataset
We use the same small SMS Spam dataset used in the previous labs. If you don't have it, run the following command in your terminal (root of the repository folder):

````bash
python scripts/download_dataset.py dbarbedillo/SMS_Spam_Multilingual_Collection_Dataset data/sms_spam
````
This will download the dataset in the `data/sms_spam` folder. You can load it with `load_from_disk(path)` providing the relative path to the downloaded dataset folder. 

**Note**: You can always use `load_dataset(url)` direcly if your notebook has access to the Internet:

````python
ds = load_dataset("dbarbedillo/SMS_Spam_Multilingual_Collection_Dataset")
````

In [ ]:
ds = load_from_disk("../../data/sms_spam")  # columns: ['labels', 'text']
df =  ds["train"].to_pandas()

df_spam = df[["labels", "text"]].copy()
df_spam.columns = ["label", "text"]
df_spam.info()

### 1.1 Curate the dataset
Before training a model, it is good practice to inspect and **clean the dataset**. Real-world datasets often contain duplicated entries, formatting inconsistencies, or missing values that can affect model performance.

In our case, duplicated messages could bias the classifier by overrepresenting certain examples. This is a standard preprocessing step in many NLP workflows.

Let’s first check whether the dataset contains duplicate messages:

In [ ]:
df_spam.duplicated().sum()

We can inspect them before deciding whether to remove them:

In [ ]:
df_spam[df_spam.duplicated(keep=False)].sort_values("text")

If we decide to remove duplicates, we can do:

In [ ]:
df_spam = df_spam.drop_duplicates()
df_spam.info()

This helps ensure that the dataset is more balanced and prevents the model from learning the same examples multiple times.

### 1.2 Inspect class distribution
In classification tasks, it is important to inspect how the examples are distributed across classes. If one class appears much more frequently than another, the dataset is said to be *imbalanced*.

Class imbalance can bias a model toward predicting the majority class more often. For example, in spam detection, if most messages are labeled as ham (not spam), a model could achieve high accuracy simply by always predicting ham, while still failing to detect spam effectively.

By checking the class distribution early, we can better understand the dataset and decide whether additional steps (resampling to a more balanced distribution) or evaluation metrics are needed.

Let’s inspect the number of examples in each class:

In [ ]:
df_spam["label"].value_counts()

## 2. Split into train/test sets
We split the dataset to estimate how well the model generalizes: we train on one partition and evaluate on unseen data; this prevents optimistic results from *testing on the training set*.

In practice, we use `train_test_split(..., test_size=0.2, random_state=42, stratify=df_spam["label"])` from sklearn. Here create an 80/20 split **stratified** by the column `label` to preserve the spam/ham ratio in both sets. The `random_state` param is to set the random seed to make the split reproducible.

Typically 80/20 is a solid default for small-medium corpora (or / and cross validation). For very large corpora, a 90/10 (or even 95/5) test set is sufficient.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_spam["text"], df_spam["label"], test_size=0.2, random_state=42, stratify=df_spam["label"]
)

print(y_train.value_counts(normalize=True))
print()
print(y_test.value_counts(normalize=True))

## 3. Represent text as Bag-of-Words
**Naïve Bayes** relies on frequency-based representations of text, where each word’s occurrence contributes evidence for a class.
We use the Bag-of-Words (BoW) model to convert messages into numerical feature vectors—each column represents a word, and each entry its count in a message.

As seen in the previous Lab, we can use the `CountVectorizer` for this task. The vectorizer is first fit on the training set to learn the vocabulary and word frequencies, and then the same vocabulary is used to transform the test set, ensuring both datasets share identical feature dimensions.

In [ ]:
vectorizer = CountVectorizer(stop_words='english')

X_train_bow = vectorizer.fit_transform(X_train) # fit to the training split
X_test_bow  = vectorizer.transform(X_test)      # transform the test split

print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")

## 4. Train the Naïve Bayes model
The Multinomial Naïve Bayes classifier learns from word frequencies to estimate how strongly each word supports a class (e.g., spam vs. ham).

We use the `MultinomialNB` class for this purpose. We can pass many parameters such as:
- alpha: value for the laplace smoothing (α = 1 by default)
- class_prior: we can override priors `P(c)` (e.g., [0.9, 0.1] ) learned from data, e.g., if you know the % of spam in real life, or want to be more conservative.

Then `.fit()` receives the training corpus in bow format `X_train_bow`, and the true labels `y_train`. Here we learn the class prior `P(c)` and conditional word probabilities `P(w | c)`.

In [ ]:
#from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB(alpha=1.0) # e.g, class_prior=[0.8, 0.2] order according to nb.classes_
nb.fit(X_train_bow, y_train)

print("Model trained.")

## 5. Evaluate the model
After training the model, we evaluate its performance on the test set, which contains messages the model has never seen before.
We use `.predict()` on the vectorized representation of the test messages. This returns the predicted class for each document -- the class with the **highest log-probability** according to the Naïve Bayes model.

### 5.1 Inspecting predictions

We use `.predict()` on the vectorized test messages. For each message, the model returns the class with the highest predicted probability according to the Naïve Bayes classifier.

The output of `.predict()` is simply an array of predicted labels:

In [ ]:
y_pred = nb.predict(X_test_bow)
y_pred

These predictions become more meaningful when we compare them with:

- the original test messages (X_test)
- the true labels (y_test)
- the predicted labels (y_pred)

Let's inspect a few examples side by side:

In [ ]:
results_df = pd.DataFrame({
    "message": X_test,
    "true_label": y_test,
    "predicted_label": y_pred
})

results_df.head(10)

### 5.2 Evaluating on the test dataset
With both the true labels (`y_test`) and the predicted labels (`y_pred`), we can now measure how well the model generalizes by comparing its predictions to the ground truth.

In [ ]:
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_estimator(nb, X_test_bow, y_test, cmap="Blues")
plt.title("Confusion Matrix – Naïve Bayes Spam Classifier")
plt.show()

**Remember**: Task matter when interpretting results. What metrics are most important to us when it comes to spam detection?

### 5.3 Inspecting class probabilities
So far, we used `.predict()` to get only the most likely class for each message -- spam or ham.
But Naïve Bayes is a probabilistic model, meaning it actually computes a full probability distribution over classes for every document.

We can access these values with `.predict_proba()`, which returns a matrix where each row corresponds to a message and each column to a class: `P(class | message)`.

In [ ]:
# Get the predicted probabilities for each class
y_proba = nb.predict_proba(X_test_bow)

# Check the class order
print("Class order:", nb.classes_)
y_proba

In [ ]:
# let's class probaility into a data frame
df_proba = pd.DataFrame(y_proba, columns=nb.classes_)
# we add true label and message to the dataframe
df_proba["true_label"] = y_test.values
df_proba["message"] = X_test.values

# Reorder columns for readability
df_proba = df_proba[["message", "true_label", "ham", "spam"]]

# Show first few rows
pd.set_option("display.max_colwidth", 100)  # so text isn't truncated
df_proba.head()

### 5.4 Adjusting the decision threshold
By default, `.predict()` labels a message as the class with the highest probability, effectively using a threshold of 0.5 in binary classification.
However, for spam detection we may want to be more conservative — for example, labeling a message as spam only if
`P(spam | d) >= τ`.

In [ ]:
# Use custom threshold τ
tau = 0.6
p_spam = y_proba[:, nb.classes_.tolist().index("spam")] # we get prob of spam
y_pred_tau = np.where(p_spam >= tau, "spam", "ham") # relabel with new tau value

print(classification_report(y_test, y_pred_tau, digits=3))

Next, let’s visualize the model’s predictions using a confusion matrix.

Normally, we could use `ConfusionMatrixDisplay.from_estimator()`, which internally computes predictions from the model. However, in this case we already created our own predictions (`y_pred_tau`) using a custom decision threshold τ. Because of this, we manually compute the confusion matrix and then pass it to `ConfusionMatrixDisplay` for visualization.

In [ ]:
# let's compute our confusion matrix with our predictions
cm = confusion_matrix(y_test, y_pred_tau, labels=nb.classes_) 

# we pass the computex matrix to display
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=nb.classes_)
disp.plot(cmap="Blues")
plt.title(f"Confusion Matrix — Naïve Bayes (τ ={tau})")
plt.show()

### 5.5 Inspect learned probabilities
Which words are most indicative of **spam** or **ham**?

After training, the Naïve Bayes model has learned how strongly each word supports one class over the other.
For every word w and class c, it stores:
- log P(w | c) — how likely the word is under that class
- log P(c) — the prior probability of that class

When classifying a new message, the model sums these log-probabilities across all words:

`log P(c) + ∑ count(w,d) × log P(w | c)`

A word is spam-indicative if it occurs much more often in spam than ham. That is:

`log P(w | spam) − log P(w | ham) > 0)`

and ham-indicative if the opposite is true.

By comparing these learned probabilities, we can see which words the model considers most characteristic of spam or ham.

In [ ]:
def print_indicative_features(nb, vectorizer, topk=10, verbose=True):
    # 1. Identify class indices (spam = 1, ham = 0)
    classes = nb.classes_.tolist()
    i_spam  = classes.index("spam")
    i_ham   = classes.index("ham")
    
    # 2. Retrieve the learned log probabilities for each word and class
    feature_names = np.array(vectorizer.get_feature_names_out())
    log_pw = nb.feature_log_prob_
    
    # 3. Compute the log-odds for each feature: spam minus ham
    log_odds = log_pw[i_spam] - log_pw[i_ham]
    
    # 4. Build a small DataFrame for inspection
    df_weights = pd.DataFrame({
        "feature": feature_names,
        "logP_w_given_spam": log_pw[i_spam], # log P(w | spam)
        "logP_w_given_ham":  log_pw[i_ham],  # log P(w | ham)
        "logodds_spam_minus_ham": log_odds,  # log P(w | spam) - log P(w | ham)
        "odds_ratio": np.exp(log_odds)  # x times more likely to be of that class
    }).sort_values("logodds_spam_minus_ham", ascending=False)
    
    # 5. Display the top indicative words for each class

    top_spam = df_weights.head(topk)                 # most spam-indicative
    top_ham  = df_weights.tail(topk).iloc[::-1]      # most ham-indicative

    if (verbose):
        print("Top spam-indicative features:")
        display(top_spam[["feature", "logodds_spam_minus_ham", "odds_ratio"]])    
        print("\nTop ham-indicative features:")
        display(top_ham[["feature", "logodds_spam_minus_ham", "odds_ratio"]])
    else:    
        spam_words = ", ".join(top_spam["feature"].tolist())
        ham_words  = ", ".join(top_ham["feature"].tolist())
        print(f"Top spam-indicative words ({topk}):\n {spam_words}")
        print()
        print(f"Top ham-indicative words  ({topk}):\n {ham_words}")
        
print_indicative_features(nb, vectorizer, topk=15, verbose=False)

## 6. Inference on new messages
We now apply the model to unseen examples to see how it combines prior + likelihoods.

In [ ]:
samples = [
    "Win a free prize today!",
    "Lunch meeting tomorrow at noon",
    "Congratulations!!! Claim your reward now",
]
X_samples = vectorizer.transform(samples)
preds = nb.predict(X_samples)
probs = nb.predict_proba(X_samples)

# Find the column index for "spam"
classes = nb.classes_.tolist()
i_spam = classes.index("spam")  # robust way

for msg, label, prob in zip(samples, preds, probs):
    print(f"{msg:50s} → {label.upper()}  (P(spam)={prob[i_spam]:.3f})")

## 7. Custom pre-processing
We have discussed in the previous lectures that pre-processing can impact the performance of the nlp tasks. 
So it is important to understand what type of pre-processing is benefitial to the specific task. There is no standard pipeline that apply to everything!


In [ ]:
import re, html, unicodedata

# we put some default parameters so we can test the impact of different decisions
def spacy_tokenizer(
    text,
    lowercase=True,
    remove_stopwords=True,
    lemmatize=True,
    remove_punctuation=True
):
    
    doc = nlp(text)
    tokens = []

    for token in doc:

        if token.is_space:
            continue

        if remove_punctuation and token.is_punct:
            continue

        if remove_stopwords and token.is_stop:
            continue

        # choose lemma or original token
        tok = token.lemma_ if lemmatize else token.text

        # optional lowercase
        if lowercase:
            tok = tok.lower()

        tokens.append(tok)

    return tokens

We can now run the pipeline with different preprocessing configurations.
To keep the interface simple, we use `partial()` to create customized versions of our tokenizer while still matching the expected `tokenizer(text)` format required by `run_nb_pipeline()`.

In [ ]:
def run_nb_pipeline(custom_tokenizer, class_prior = None, signal_tokens = None):
    """Train NB. class_prior = [P(ham), P(spam)] or None to learn from data."""
    vectorizer = CountVectorizer(tokenizer=custom_tokenizer, stop_words=None, token_pattern=None)
    X_train_bow = vectorizer.fit_transform(X_train)
    X_test_bow  = vectorizer.transform(X_test)
    
    print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")

    if signal_tokens != None:
        vocab = vectorizer.vocabulary_

        for token in signal_tokens:
            if token in vocab:
                idx = vocab[token]
                count = X_train_bow[:, idx].sum()
                print(f"{token}: {count}")
            else:
                print(f"{token}: not in vocabulary")
    
    nb = MultinomialNB(alpha=1.0, class_prior=class_prior)
    nb.fit(X_train_bow, y_train)
    
    y_pred = nb.predict(X_test_bow)
    print(classification_report(y_test, y_pred))
    
    ConfusionMatrixDisplay.from_estimator(nb, X_test_bow, y_test, cmap="Blues")
    plt.title("Confusion Matrix – Naïve Bayes Spam Classifier with custom pre-processing ")
    plt.show()

    # We print out the top indicative words
    print_indicative_features(nb, vectorizer, topk=300, verbose=False)

    return (nb, vectorizer, y_pred)

# configure the tokeniser and run the pipeline!    
from functools import partial

# Create reusable tokenizer configurations
# partial creates a function with arguments already prefilled
spacy_tokenizer_custom = partial(
    spacy_tokenizer,
    lowercase=True,
    remove_stopwords=True,
    lemmatize=True,
    remove_punctuation=True
)

nb_a, vec_a, y_pred_a = run_nb_pipeline(spacy_tokenizer_custom)

### Reflect
- What was the effect of using more aggressive normalization?
- Which preprocessing configuration performed best?
- Which information may have been lost during preprocessing?
- Were there any preprocessing steps that unexpectedly hurt performance?

## 8. Custom features
We can also add more than "word" features to NB. There are different ways to do so, a simpler one being injecting special tokens into NB. For example, we can have a features such as
- `__HAS_EXCLAM__` if the given message has a number of excalamation points !!
- `__MANY_CAPS__` if the messages have more than a given number of uppercase words

We can also clean our text further by collapsing certain patterns that are spelled differently. For example, we can collapse all variations of "terms and conditions", such as "t&c", "ts&cs" that are featured in the dataset and are currently counted as separate tokens. 

We also have numbers being counted separately as individual tokens, e.g., '500', '180', so we can characterise numbers, phone numbers, etc.

In [ ]:
import re
import html
from spacy.symbols import ORTH
from functools import partial

# Simple signal patterns
URL_RE = re.compile(r"https?://\S+|www\.\S+|\S+\.(com|org|net|de)\S*", re.I)
PHONE_RE = re.compile(r"\+?\d[\d\s\-()]{7,}")
PREMIUM_PHONE_RE = re.compile(r"\b0871\d+\b")
EXCLAM_RE = re.compile(r"!{2,}")
TNC_RE = re.compile(r"\b(ts?&cs?|terms|conditions)\b", re.I)
TAG_RE = re.compile(r"<[^>]+>")

In [ ]:
SIGNAL_TOKENS = [
    "__HAS_URL__",
    "__HAS_PHONE__",
    "__HAS_PREMIUM_PHONE__",
    "__HAS_EXCLAM__",
    "__HAS_TNC__",
    "__MANY_CAPS__"
]

for token in SIGNAL_TOKENS:
    nlp.tokenizer.add_special_case(token, [{ORTH: token}])

In [ ]:
def clean_text(text):
    """
    Minimal text cleanup before tokenization.
    """
    text = html.unescape(text)
    text = TAG_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
def append_signal_tokens(text, caps_threshold=3):
    """
    Add simple task-specific features as extra tokens.
    """
    signals = []

    if URL_RE.search(text):
        signals.append("__HAS_URL__")

    if PREMIUM_PHONE_RE.search(text):
        signals.append("__HAS_PREMIUM_PHONE__")

    if PHONE_RE.search(text):
        signals.append("__HAS_PHONE__")

    if EXCLAM_RE.search(text):
        signals.append("__HAS_EXCLAM__")

    if TNC_RE.search(text):
        signals.append("__HAS_TNC__")

    caps_count = sum(
        1 for word in text.split()
        if word.isupper() and len(word) >= 2
    )

    if caps_count >= caps_threshold:
        signals.append("__MANY_CAPS__")

    return signals

def handle_signal_tokens(text, signal_mode="append"):
    """
    Add signal tokens to the text.

    signal_mode:
    - "none": do not use signal tokens
    - "append": keep the original text and add signal tokens at the end
    - "replace": replace matched patterns with signal tokens
    """
    if signal_mode == "none":
        return text, []

    if signal_mode == "append":
        signal_tokens = append_signal_tokens(text)
        return text, signal_tokens

    if signal_mode == "replace":
        text = URL_RE.sub(" __HAS_URL__ ", text)
        text = PREMIUM_PHONE_RE.sub(" __HAS_PREMIUM_PHONE__ ", text)
        text = PHONE_RE.sub(" __HAS_PHONE__ ", text)
        text = EXCLAM_RE.sub(" __HAS_EXCLAM__ ", text)
        text = TNC_RE.sub(" __HAS_TNC__ ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text, []

    raise ValueError("signal_mode must be 'none', 'append', or 'replace'")    

In [ ]:
def spacy_tokenizer_with_signals(
    text,
    lowercase=True,
    remove_stopwords=False,
    lemmatize=False,
    remove_punctuation=True,
    signal_mode="append"
):
    """
    Configurable tokenizer with optional task-specific signal tokens.
    """
    text = clean_text(text)
    text, signal_tokens = handle_signal_tokens(text, signal_mode=signal_mode)

    doc = nlp(text)
    tokens = []

    for token in doc:
        if token.is_space:
            continue

        if token.text.startswith("__") and token.text.endswith("__"):
            tokens.append(token.text)
            continue

        if remove_punctuation and token.is_punct:
            continue

        if remove_stopwords and token.is_stop:
            continue

        tok = token.lemma_ if lemmatize else token.text

        if lowercase:
            tok = tok.lower()

        tokens.append(tok)

    return tokens + signal_tokens

In [ ]:
tokenizer_minimal = partial(
    spacy_tokenizer_with_signals,
    lowercase=True,
    remove_stopwords=False,
    lemmatize=False,
    remove_punctuation=True,
    signal_mode="none"
)

tokenizer_with_signals = partial(
    spacy_tokenizer_with_signals,
    lowercase=True,
    remove_stopwords=True,
    lemmatize=False,
    remove_punctuation=True,
    signal_mode="append"
)

In [ ]:
tokenizer_minimal("You have won a FREE vacation!!! t&c apply")

In [ ]:
nb_a, vec_a, y_pred_a = run_nb_pipeline(tokenizer_minimal)

nb_b, vec_b, y_pred_b = run_nb_pipeline(tokenizer_with_signals, signal_tokens=SIGNAL_TOKENS)


### Inspect errors
The following helper functions let you analyze misclassified messages and understand why the model made a certain prediction. Qualitatively analysing the output of your model is a good way of understanding if there are patters of errors that can be addressed.

Use `preview_errors_explained()` to scan all test samples, providing the true label you want to explore. Then it selects the messages that were misclassified and displays a table with:

- **index**: original row index of the message in the dataset
- **true**: the correct label of the message
- **pred**: the label predicted by the classifier
- **p_spam**: predicted probability that the message is spam
- **log_odds**: overall spam-vs-ham evidence score in log-space (positive = more spam-like, negative = more ham-like)
- **top_pos**: words contributing most strongly toward the spam prediction
- **top_neg**: words contributing most strongly toward the ham prediction
- **prior_log_odds**: contribution of the class prior alone before considering the words in the message
- **text**: the original message text being analyzed



In [ ]:
from utils import preview_errors_explained

print("— Errors with simple processing —")
df_err_A = preview_errors_explained(X_test, y_test, y_pred_a, nb_a, vec_a, true_label="spam")

In [ ]:
print("— Errors with hybrid processing —")
df_err_B = preview_errors_explained(
    X_test, y_test, y_pred_b, nb_b, vec_b, true_label="spam"
)

## 9. Discussion
- What is the effect of normalising (pre-processing) the text? Try normalising, not normalising and reflect on the results.
- Reflect on the performance of the model, and engineered features. What do we gain and lose with your implementation of more advanced pre-processing?